# 🎯 Model Training & Evaluation

Denne notebook træner og evaluerer alle fire xG-modeller på vores datasæt fra Superliga 2024/25.

Vi bruger det samme 80/20 stratified split, den samme preprocessing og de samme features til alle modeller — så sammenligningen er fair.

**Modeller:**
- 📐 Logistic Regression — simpel lineær baseline
- 🌲 XGBoost — gradient boosting benchmark
- 🤖 TabPFN — pretrained transformer (ingen træning på vores data)
- 🧠 MLP — neural network (hjælpemodel)

**Output:** `outputs/model_results.csv` med AUC og log-loss for hver model

## 1. Imports & setup

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss, roc_auc_score, balanced_accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler

DATA_PATH = Path("../../../outputs/outputs/all_matches_features_final.csv")
OUT_PATH  = Path("../../../outputs/model_results.csv")

print("Imports OK ✓")

## 2. Features

Vi bruger alle 15 features — 2 shot geometry og 13 kontekstuelle features fra tracking data.

In [2]:
FEATURES = [
    # Shot geometry
    "distance_m",
    "angle_rad",
    # Kontekst — event data
    "goal_diff",
    "time_since_last_event_s",
    "shot_body_part",
    "play_pattern",
    # Kontekst — tracking only
    "pressure_nd_dist_m",
    "pressure_def_count_r1m",
    "pressure_def_count_r2m",
    "obstruction_count",
    "gk_ball_distance",
    "gk_depth",
    "gk_lateral_offset",
    "ball_speed_mps",
    "shooter_speed_mps",
]

print(f"Antal features: {len(FEATURES)}")

Antal features: 15


## 3. Load data

In [3]:
df = pd.read_csv(DATA_PATH)
df["is_goal"] = pd.to_numeric(df["is_goal"], errors="coerce")
df = df.loc[df["is_goal"].isin([0, 1])].reset_index(drop=True)

print(f"Totalt antal skud : {len(df)}")
print(f"Mål               : {int(df['is_goal'].sum())} ({df['is_goal'].mean()*100:.1f}%)")
print(f"Kampe             : {df['game_id'].nunique()}")

Totalt antal skud : 4957
Mål               : 590 (11.9%)
Kampe             : 189


## 4. Preprocessing

**Trin:**
1. Encode kategoriske variable (fx `shot_body_part`, `play_pattern`) til tal
2. Split 80/20 — `stratify=y` sikrer samme goal-rate i begge sæt
3. Imputer manglende værdier med **trænings-median** (aldrig test-median — det ville give data leakage)
4. Standardiser features med `StandardScaler` (til LogReg og MLP)

In [4]:
# Encode kategoriske variable
def encode_features(df_sub):
    X = df_sub.copy()
    for col in X.columns:
        if X[col].dtype == object or str(X[col].dtype) == "category":
            codes, _ = pd.factorize(X[col])
            X[col] = codes.astype(float)
            X.loc[X[col] == -1, col] = np.nan
        else:
            X[col] = pd.to_numeric(X[col], errors="coerce")
    return X.to_numpy(dtype=float)

# Imputer med trænings-median (ingen data leakage)
def impute_with_train_median(X_train, X_test):
    medians = np.nanmedian(X_train, axis=0)
    for j in range(X_train.shape[1]):
        X_train[np.isnan(X_train[:, j]), j] = medians[j]
        X_test[np.isnan(X_test[:, j]), j]   = medians[j]
    return X_train, X_test

available = [f for f in FEATURES if f in df.columns]
X_raw = encode_features(df[available])
y     = df["is_goal"].astype(int).to_numpy()

# 80/20 stratified split — random_state=42 giver reproducerbart split
X_train, X_test, y_train, y_test = train_test_split(
    X_raw, y, test_size=0.2, stratify=y, random_state=42
)
print(f"Træning : {len(X_train)} skud")
print(f"Test    : {len(X_test)} skud")

X_train, X_test = impute_with_train_median(X_train.copy(), X_test.copy())

# Standardiser (kun til LogReg og MLP)
scaler     = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

# Class imbalance vægte (til XGBoost og MLP)
n_neg = int((y_train == 0).sum())
n_pos = int((y_train == 1).sum())
spw   = n_neg / max(n_pos, 1)
sample_weights = np.where(y_train == 1, spw, 1.0)

print(f"\nClass imbalance ratio (scale_pos_weight): {spw:.2f}")

Træning : 3965 skud
Test    : 992 skud

Class imbalance ratio (scale_pos_weight): 7.40


## 5. Træn modeller

Alle modeller trænes på de samme 3.965 træningsskud og evalueres på de samme 992 testskud.

### 📐 Logistic Regression

Lineær model der lærer en koefficient per feature. Features er standardiseret så koefficienterne kan sammenlignes direkte. `class_weight='balanced'` håndterer at mål er sjældne (~12%).

In [ ]:
lr = LogisticRegression(C=1.0, max_iter=1000, class_weight="balanced", random_state=42)
lr.fit(X_train_sc, y_train)
p_lr = lr.predict_proba(X_test_sc)[:, 1]

auc_lr = round(float(roc_auc_score(y_test, p_lr)), 4)
ll_lr  = round(float(log_loss(y_test, p_lr)), 4)
ba_lr  = round(float(balanced_accuracy_score(y_test, (p_lr >= 0.5).astype(int))), 4)
print(f"Logistic Regression — AUC: {auc_lr} | Log-loss: {ll_lr} | Balanced Acc: {ba_lr}")

### 🌲 XGBoost

Gradient boosted trees der fanger ikke-lineære sammenhænge og feature-interaktioner. `scale_pos_weight` kompenserer for class imbalance. Ingen feature-skalering nødvendig.

In [ ]:
from xgboost import XGBClassifier

xgb = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=spw,
    eval_metric="logloss",
    random_state=42,
    verbosity=0,
)
xgb.fit(X_train, y_train)
p_xgb = xgb.predict_proba(X_test)[:, 1]

auc_xgb = round(float(roc_auc_score(y_test, p_xgb)), 4)
ll_xgb  = round(float(log_loss(y_test, p_xgb)), 4)
ba_xgb  = round(float(balanced_accuracy_score(y_test, (p_xgb >= 0.5).astype(int))), 4)
print(f"XGBoost — AUC: {auc_xgb} | Log-loss: {ll_xgb} | Balanced Acc: {ba_xgb}")

### 🤖 TabPFN

TabPFN er en transformer pretrænet på syntetiske tabular datasæt. Den **trænes ikke** på vores data — i stedet caches træningsdataene som kontekst i modellens attention mechanism (`fit_with_cache`). Forudsigelserne genereres i ét forward pass. Ingen hyperparameter-tuning eller feature-skalering nødvendig.

In [ ]:
from tabpfn import TabPFNClassifier

tabpfn = TabPFNClassifier(fit_mode="fit_with_cache", random_state=42)
tabpfn.fit(X_train, y_train)
p_tab = tabpfn.predict_proba(X_test)[:, 1]

auc_tab = round(float(roc_auc_score(y_test, p_tab)), 4)
ll_tab  = round(float(log_loss(y_test, p_tab)), 4)
ba_tab  = round(float(balanced_accuracy_score(y_test, (p_tab >= 0.5).astype(int))), 4)
print(f"TabPFN — AUC: {auc_tab} | Log-loss: {ll_tab} | Balanced Acc: {ba_tab}")

### 🧠 MLP (Neural Network)

To skjulte lag (64 → 32 neuroner) med ReLU aktivering. Features standardiseres inden træning. Class imbalance håndteres via `sample_weight` da `MLPClassifier` ikke understøtter `class_weight` direkte.

In [ ]:
mlp = MLPClassifier(
    hidden_layer_sizes=(64, 32),
    activation="relu",
    max_iter=1000,
    random_state=42,
)
mlp.fit(X_train_sc, y_train, sample_weight=sample_weights)
p_mlp = mlp.predict_proba(X_test_sc)[:, 1]

auc_mlp = round(float(roc_auc_score(y_test, p_mlp)), 4)
ll_mlp  = round(float(log_loss(y_test, p_mlp)), 4)
ba_mlp  = round(float(balanced_accuracy_score(y_test, (p_mlp >= 0.5).astype(int))), 4)
print(f"MLP — AUC: {auc_mlp} | Log-loss: {ll_mlp} | Balanced Acc: {ba_mlp}")

## 6. Resultater

Samlet oversigt og gem til CSV.

In [ ]:
results = pd.DataFrame([
    {"model": "LogReg",  "auc": auc_lr,  "log_loss": ll_lr,  "balanced_acc": ba_lr},
    {"model": "XGBoost", "auc": auc_xgb, "log_loss": ll_xgb, "balanced_acc": ba_xgb},
    {"model": "TabPFN",  "auc": auc_tab, "log_loss": ll_tab, "balanced_acc": ba_tab},
    {"model": "MLP",     "auc": auc_mlp, "log_loss": ll_mlp, "balanced_acc": ba_mlp},
])

print(results.to_string(index=False))

OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
results.to_csv(OUT_PATH, index=False)
print(f"\nGemt → {OUT_PATH} ✓")

In [ ]:
# Feature ablation — sammenlign TabPFN med 15, 11 (fjern 4) og 10 (fjern 5) features
from tabpfn import TabPFNClassifier

# 15 features (fuld model)
FEATS_15 = [
    "distance_m", "angle_rad", "goal_diff", "time_since_last_event_s",
    "shot_body_part", "play_pattern", "pressure_nd_dist_m",
    "pressure_def_count_r1m", "pressure_def_count_r2m", "obstruction_count",
    "gk_ball_distance", "gk_depth", "gk_lateral_offset",
    "ball_speed_mps", "shooter_speed_mps",
]

# 11 features — fjern 4 (goal_diff, time_since_last_event_s, pressure_def_count_r1m, pressure_def_count_r2m)
# gk_depth beholdes da beeswarm viser den har betydning
FEATS_11 = [
    "distance_m", "angle_rad",
    "shot_body_part", "play_pattern", "pressure_nd_dist_m",
    "obstruction_count", "gk_ball_distance", "gk_depth", "gk_lateral_offset",
    "ball_speed_mps", "shooter_speed_mps",
]

# 10 features — fjern 5 (+ gk_depth)
FEATS_10 = [
    "distance_m", "angle_rad",
    "shot_body_part", "play_pattern", "pressure_nd_dist_m",
    "obstruction_count", "gk_ball_distance", "gk_lateral_offset",
    "ball_speed_mps", "shooter_speed_mps",
]

ablation_results = []
for label, feats in [("15 features", FEATS_15), ("11 features (fjern 4)", FEATS_11), ("10 features (fjern 5)", FEATS_10)]:
    avail = [f for f in feats if f in df.columns]
    X_abl = encode_features(df[avail])
    X_tr, X_te, y_tr, y_te = train_test_split(X_abl, y, test_size=0.2, stratify=y, random_state=42)
    X_tr, X_te = impute_with_train_median(X_tr.copy(), X_te.copy())

    m = TabPFNClassifier(fit_mode="fit_with_cache", random_state=42)
    m.fit(X_tr, y_tr)
    p = m.predict_proba(X_te)[:, 1]

    auc_ = round(float(roc_auc_score(y_te, p)), 4)
    ll_  = round(float(log_loss(y_te, p)), 4)
    ba_  = round(float(balanced_accuracy_score(y_te, (p >= 0.5).astype(int))), 4)
    ablation_results.append({"config": label, "auc": auc_, "log_loss": ll_, "balanced_acc": ba_})
    print(f"TabPFN {label:25s} — AUC: {auc_} | Log-loss: {ll_} | Balanced Acc: {ba_}")

print()
pd.DataFrame(ablation_results)